In [107]:
# autoreload
%load_ext autoreload
%autoreload 2

from openai import OpenAI
from tqdm import tqdm
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Any
from collections.abc import Callable


from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

import sys
sys.path.append("../../02_vector_search")
sys.path.append("../utils")
from embedder import Embedder
from evaluation_utils import llm_structured_retry

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [108]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_agentic_rag/.env")
# Check environment variables
import os
required_env_vars = ["OPENAI_API_KEY"]
missing_env_vars = [var for var in required_env_vars if var not in os.environ]
if missing_env_vars:
    raise EnvironmentError(f"Missing required environment variables: {', '.join(missing_env_vars)}")

### Functions

In [109]:
def text_search(index, query, num_results=5, filter_dict=None, boost_dict=None):
    """
    Search the index for a given query.

    Args:
        index: The minsearch Index instance.
        query (str): The search query.
        num_results (int): The number of results to return. Default is 5.
        filter_dict (dict): A dictionary of filters to apply to the search. Default is None.
        boost_dict (dict): A dictionary of fields to boost in the search. Default is None

    Returns:
        list: A list of search results.
    """
    return index.search(
        query,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )

In [110]:
def vector_search(index, embedding, query, num_results=5):
    """
    Search the index for a given query.

    Args:
        index: The minsearch VectorSearch instance.
        embedding: The embedding model instance.
        query (str): The search query.
        num_results (int): The number of results to return. Default is 5.

    Returns:
        list: A list of search results.
    """
    query_vector = embedding.encode(query)
    return index.search(query_vector, num_results=num_results)


In [111]:
def rrf(result_lists, k=60, num_results=5):
    """
    Fuse multiple ranked result lists using Reciprocal Rank Fusion (RRF).

    Each document gets a fused score computed as:

        score(doc) = sum(1 / (k + rank))

    where `rank` is 1-based within each input list and `k` controls how
    strongly top-ranked results are favored.

    Args:
        result_lists (list[list[dict]]): Ranked lists of documents.
            Each document is expected to contain at least:
            - "filename" (str)
            - "start" (int)
        k (int, optional): RRF constant. Higher values reduce rank impact.
            Defaults to 60.
        num_results (int, optional): Number of fused results to return.
            Defaults to 5.

    Returns:
        list[dict]: Top fused documents, ordered by descending fused score.
    """
    if num_results <= 0:
        return []
    if k < 0:
        raise ValueError("k must be >= 0")

    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results, start=1):  # 1-based rank
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
            docs[key] = doc

    ranked_items = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    return [docs[key] for key, _ in ranked_items[:num_results]]

In [112]:
def hybrid_search(query: str, k: int = 60) -> list[dict]:
    """
    Run hybrid retrieval by combining text and vector search with RRF.
    Expects `text_index`, `vector_index`, and `embedding` to be defined in the notebook.
    """
    text_results = text_search(
        index=text_index,
        query=query,
        num_results=10,
    )

    vector_results = vector_search(
        index=vector_index,
        embedding=embedder,
        query=query,
        num_results=10,
    )

    return rrf([text_results, vector_results], k=k, num_results=5)


In [118]:
# Helper function to compute relevance for a single query
def _compute_relevance(
    q: dict[str, str],
    search_function: Callable[..., list[dict[str, Any]]],
    query_key: str = "question",
    doc_key: str = "filename",
    **search_kwargs: Any,
) -> list[int]:
    """
    Compute binary relevance labels for one query.

    Args:
        q: Ground-truth record with at least query and expected document fields.
        search_function: Retrieval function that accepts `query=` and returns ranked docs.
        query_key: Key used to read the query text from `q`.
        doc_key: Key used to compare expected and retrieved document IDs.
        **search_kwargs: Extra keyword arguments forwarded to `search_function`.

    Returns:
        A list of 0/1 relevance labels aligned with retrieved ranking positions.
    """
    query = q[query_key]
    expected_doc = q[doc_key]
    results = search_function(query=query, **search_kwargs)

    return [int(d[doc_key] == expected_doc) for d in results]


# Compute the relevance for all ground truth records
def compute_relevance_total(
    ground_truth: list[dict[str, str]],
    search_function: Callable[..., list[dict[str, Any]]],
    **search_kwargs: Any,
) -> list[list[int]]:
    """
    Compute relevance labels for all ground-truth queries.

    Args:
        ground_truth: List of query-document ground-truth records.
        search_function: Retrieval function used to generate ranked results per query.
        **search_kwargs: Extra keyword arguments forwarded to `search_function`.

    Returns:
        A 2D relevance matrix (rows = queries, columns = ranked results).
    """
    relevance_total: list[list[int]] = []

    for q in tqdm(ground_truth):
        relevance = _compute_relevance(q, search_function, **search_kwargs)
        relevance_total.append(relevance)

    return relevance_total

In [114]:
def hit_rate(relevance: list[list[int | bool]] | np.ndarray) -> float:
    """
    Compute hit rate as the percentage of queries with at least one relevant result.

    Args:
        relevance: A 2D relevance matrix where each row corresponds to one query
            and each value indicates whether a retrieved document is relevant.

    Returns:
        Hit rate as a percentage in the range [0, 100].
    """
    relevance_array = np.array(relevance)
    return float(np.mean(relevance_array.max(axis=1)))


In [115]:
def mrr(relevance):
    relevance_array = np.array(relevance)
    reciprocal_ranks = 1 / (np.arange(relevance_array.shape[1]) + 1)
    mrr_array = relevance_array * reciprocal_ranks
    return float(np.sum(mrr_array) / relevance_array.shape[0])

In [121]:
def evaluate(ground_truth, search_function, **search_kwargs):
    relevance_total = compute_relevance_total(
        ground_truth,
        search_function,
        **search_kwargs
    )

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

### 1. Retrieve the data from the GitHub repository and parse it into a list of documents

In [68]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [69]:
# Number of documents
len(documents)

72

# 2. Prompt for test questions that are answered in the lessons

In [70]:
# First 3 pages
documents[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [71]:
# Format questions output
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [72]:

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


In [73]:
# Instantiate client
openai_client = OpenAI()

In [74]:
# Function to generate json for each doc, call llm and create the recor
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [75]:
# test for first 3 documents
ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:07<00:00,  2.37s/it]


Question 1 What's the average number of input tokens across these 3 calls?

In [76]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=110, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1130),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=116, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1402),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=86, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1839)]

In [77]:
mean_input_tokens = np.mean([ResponseUsage.input_tokens for ResponseUsage in usages])
mean_input_tokens

np.float64(1353.0)

# Searching the chunks

In [93]:
# Load full ground truth from file
ground_truth = pd.read_csv("../data/ground-truth.csv").to_dict(orient="records")

In [95]:
ground_truth

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module?',
  'filename': '01-agentic-rag/lessons/02-environmen

In [80]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [81]:
len(chunks)

295

In [87]:
# Embed the chunks with encode batch
embedder = Embedder(path="../../02_vector_search/models/Xenova/all-MiniLM-L6-v2")
embeddings = embedder.encode_batch([chunk.get("content") for chunk in chunks])

In [89]:
# Built text and vector indexes
text_index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"],)
# Fit the index with the documents
text_index.fit(chunks)

vector_index = vindex = VectorSearch(keyword_fields=["course"])
vector_index.fit(embeddings, chunks)

Question 2 After running text_search for it, what's the filename of the first result?

In [96]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [97]:
text_results = text_search(
    index=text_index,
    query=q,
    num_results=5,
)

In [100]:
text_results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

Question 3 After vector search what's the filename of the first result?

In [101]:
vector_results = vector_search(
    index=vector_index,
    embedding=embedder,
    query=q,
    num_results=5,
)

In [102]:
vector_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

# Evaluating search

Question 4 Whats the hit rate for text search

In [123]:
evaluate(ground_truth, text_search, index=text_index, query_key="question", doc_key="filename")

  0%|          | 0/360 [00:00<?, ?it/s]

100%|██████████| 360/360 [00:00<00:00, 505.81it/s]


{'hit_rate': 75.83333333333333, 'mrr': np.float64(83.22222222222223)}

Question 5 What is the MRR for vector search

In [125]:
evaluate(ground_truth, vector_search, index=vector_index, query_key="question", doc_key="filename", embedding=embeddings)

  0%|          | 0/360 [00:00<?, ?it/s]


AttributeError: 'numpy.ndarray' object has no attribute 'encode'